# Building a Feature Store: Offline + Online Layers

This notebook is the hands-on exercise for the session on **Feature Engineering & Feature Store Architecture**.

We will use **Feast** (open source) to build a small but complete feature store and plug it into a model pipeline.

## Plan

1. Generate a synthetic customer-transactions dataset and engineer a few features
2. Define entities and feature views (this is the "define once" step)
3. Register the feature definitions with Feast
4. Fetch **historical** features for training (point-in-time correct join, offline store)
5. Train a simple model
6. **Materialize** features into the online store
7. Fetch **online** features to simulate real-time serving
8. Compare offline vs. online values to confirm consistency
9. Stretch goal: intentionally introduce a training-serving skew bug and detect it

No ML algorithm here is the point — the point is the **infrastructure** around the features.

## Step 0: Setup

Feast needs to be installed. We will use the local/file-based setup:
- **Offline store**: a Parquet file (stand-in for a data warehouse)
- **Online store**: SQLite (stand-in for Redis/DynamoDB in production)

This mirrors real feature store architecture without requiring any external infrastructure.

In [1]:
# Install Feast (only needs to run once per environment)
%pip install feast --quiet

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

Note: you may need to restart the kernel to use updated packages.


In [26]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Fix the random seed so the notebook is reproducible
np.random.seed(42)

# This is the folder Feast will treat as our "feature repository"
REPO_PATH = "feature_repo"
os.makedirs(f"{REPO_PATH}/data", exist_ok=True)

print("Feature repo folder ready at:", os.path.abspath(REPO_PATH))

Feature repo folder ready at: C:\Shridhar\Study\BITS\Week 9\Demo\feature_repo


## Step 1: Generate Data and Engineer Features

We simulate 60 days of transactions for 200 customers, then engineer a few features per customer per day:

- `txn_amount_sum` — total amount spent that day
- `txn_count` — number of transactions that day
- `avg_txn_7d` — rolling 7-day average spend (a classic "requires history" feature)

In a real pipeline, this aggregation step would run on Spark/SQL against a data warehouse. Here we do it in pandas to keep the exercise self-contained — the feature store concepts are identical regardless of the compute engine.

In [29]:
n_customers = 200
n_days = 60
rows = []
start = datetime.now() - timedelta(days=n_days)

for day in range(n_days):
    ts = start + timedelta(days=day)
    for cust in range(n_customers):
        # not every customer transacts every day
        if np.random.rand() < 0.3:
            rows.append({
                "customer_id": cust,
                "event_timestamp": ts,
                "txn_amount": max(5, np.random.normal(50, 20)),
                "txn_count_1d": np.random.poisson(2),
            })

raw_transactions = pd.DataFrame(rows)
print("Raw transaction rows:", raw_transactions.shape)
raw_transactions.head()

Raw transaction rows: (3600, 4)


,customer_id,event_timestamp,txn_amount,txn_count_1d
0,4,2026-05-16 20:53:05.824646,55.580826,1
1,7,2026-05-16 20:53:05.824646,70.210306,1
2,11,2026-05-16 20:53:05.824646,29.743378,1
3,14,2026-05-16 20:53:05.824646,56.284947,2
4,16,2026-05-16 20:53:05.824646,37.987226,2


In [31]:
# Aggregate raw transactions into daily per-customer features
raw_transactions = raw_transactions.sort_values(["customer_id", "event_timestamp"])

customer_features = raw_transactions.groupby(["customer_id", "event_timestamp"]).agg(
    txn_amount_sum=("txn_amount", "sum"),
    txn_count=("txn_count_1d", "sum"),
).reset_index()

# Rolling 7-day average spend per customer (a feature that needs history, not just "today")
customer_features["avg_txn_7d"] = (
    customer_features.groupby("customer_id")["txn_amount_sum"]
    .transform(lambda s: s.rolling(7, min_periods=1).mean())
)

# Feast needs a "created" timestamp to know when this row was written (for point-in-time joins)
customer_features["created"] = customer_features["event_timestamp"]

print("Engineered feature rows:", customer_features.shape)
customer_features.head()

Engineered feature rows: (3600, 6)


,customer_id,event_timestamp,txn_amount_sum,txn_count,avg_txn_7d,created
0,0,2026-05-20 20:53:05.824646,10.672868,2,10.672868,2026-05-20 20:53:05.824646
1,0,2026-05-23 20:53:05.824646,67.169022,1,38.920945,2026-05-23 20:53:05.824646
2,0,2026-05-30 20:53:05.824646,50.290180,6,42.710690,2026-05-30 20:53:05.824646
3,0,2026-06-05 20:53:05.824646,63.245260,1,47.844333,2026-06-05 20:53:05.824646
4,0,2026-06-09 20:53:05.824646,65.971333,2,51.469733,2026-06-09 20:53:05.824646


In [33]:
# Save as Parquet — this file plays the role of our offline store's data source
# (in production this would be a table in BigQuery / Redshift / a data lake)
customer_features.to_parquet(f"{REPO_PATH}/data/customer_features.parquet", index=False)
print("Saved offline source parquet file.")

Saved offline source parquet file.


## Step 2: Define the Feature Repository

This is the "define once" step that prevents training-serving skew. We declare:

- An **Entity** (`customer_id`) — the object our features describe
- A **FileSource** — where the raw feature data lives (our Parquet file)
- A **FeatureView** — the group of features tied to that entity and source, with a time-to-live (TTL)

Feast writes these definitions to a **registry**, which becomes the single source of truth that both the offline and online paths read from.

In [36]:
from datetime import timedelta
from feast import FeatureStore, Entity, FeatureView, Field, FileSource
from feast.types import Int64, Float32

# The entity: what the features are "about"
customer = Entity(name="customer_id", description="Unique customer identifier")

# Where the feature values physically live (offline source)
customer_source = FileSource(
    path=os.path.abspath(f"{REPO_PATH}/data/customer_features.parquet"),
    timestamp_field="event_timestamp",
    created_timestamp_column="created",
)

# The feature view: features + entity + source + TTL, all in one definition
customer_txn_stats = FeatureView(
    name="customer_txn_stats",
    entities=[customer],
    ttl=timedelta(days=90),
    schema=[
        Field(name="txn_amount_sum", dtype=Float32),
        Field(name="txn_count", dtype=Int64),
        Field(name="avg_txn_7d", dtype=Float32),
    ],
    source=customer_source,
    online=True,
)

print("Entity and feature view defined.")

Entity and feature view defined.


C:\Users\galan\AppData\Local\Temp\ipykernel_5708\3053991003.py:6: DeprecationWarning: Entity value_type will be mandatory in the next release. Please specify a value_type for entity 'customer_id'.
  customer = Entity(name="customer_id", description="Unique customer identifier")


In [38]:
# Write feature_store.yaml — this tells Feast where the registry and online store live.
# provider: local  -> no cloud infra needed, everything runs on this machine
yaml_content = f"""project: feature_store_demo
provider: local
registry: data/registry.db
online_store:
    type: sqlite
    path: data/online_store.db
entity_key_serialization_version: 3
"""

with open(f"{REPO_PATH}/feature_store.yaml", "w") as f:
    f.write(yaml_content)

print(yaml_content)

project: feature_store_demo
provider: local
registry: data/registry.db
online_store:
    type: sqlite
    path: data/online_store.db
entity_key_serialization_version: 3



In [40]:
# Register (apply) our definitions to the Feast registry.
# This is the equivalent of running `feast apply` from the CLI, done here via the SDK.
store = FeatureStore(repo_path=REPO_PATH)
store.apply([customer, customer_txn_stats])

print("Registered feature views:")
for fv in store.list_feature_views():
    print(" -", fv.name, "->", [f.name for f in fv.features])

Registered feature views:
 - customer_txn_stats -> ['txn_amount_sum', 'txn_count', 'avg_txn_7d']


## Step 3: Fetch Historical Features for Training

For training, we need **point-in-time correct** features: for each row in our entity dataframe (a customer + a timestamp), Feast looks up the most recent feature values *as of that timestamp* — never using data from the future. This is exactly the join that prevents the leakage bug we saw in the slides.

In [42]:
# Build an entity dataframe: which customer, at which point in time, do we want features for?
entity_df = customer_features[["customer_id", "event_timestamp"]].drop_duplicates()

training_df = store.get_historical_features(
    entity_df=entity_df,
    features=[
        "customer_txn_stats:txn_amount_sum",
        "customer_txn_stats:txn_count",
        "customer_txn_stats:avg_txn_7d",
    ],
).to_df()

print("Training dataframe shape:", training_df.shape)
training_df.head()

Training dataframe shape: (3600, 5)


,customer_id,event_timestamp,txn_amount_sum,txn_count,avg_txn_7d
0,45,2026-05-16 20:53:05.824646+00:00,70.849832,0,70.849832
1,190,2026-05-16 20:53:05.824646+00:00,56.483327,4,56.483327
2,108,2026-05-16 20:53:05.824646+00:00,69.543600,4,69.543600
3,132,2026-05-16 20:53:05.824646+00:00,60.275719,1,60.275719
4,46,2026-05-16 20:53:05.824646+00:00,60.028967,2,60.028967


## Step 4: Train a Simple Model

The model itself is not the point of this exercise — a simple classifier is enough to show the pipeline works end-to-end using features pulled from the store.

We create a synthetic label (`high_spender`) just so we have something to predict.

In [44]:
from sklearn.linear_model import LogisticRegression

training_df["high_spender"] = (
    training_df["avg_txn_7d"] > training_df["avg_txn_7d"].median()
).astype(int)

feature_cols = ["txn_amount_sum", "txn_count", "avg_txn_7d"]
X_train = training_df[feature_cols].fillna(0)
y_train = training_df["high_spender"]

model = LogisticRegression()
model.fit(X_train, y_train)

print("Train accuracy:", model.score(X_train, y_train))

Train accuracy: 0.9986111111111111


## Step 5: Materialize Features into the Online Store

`materialize_incremental` pushes the latest feature values from the offline source into the online store (SQLite here, Redis/DynamoDB in production). This is the step that makes features available for **low-latency lookups at serving time**.

In [46]:
store.materialize_incremental(end_date=datetime.now())
print("Materialization complete.")

Materializing 1 feature views to 2026-07-15 20:53:21+00:00 into the sqlite online store.

customer_txn_stats from 2026-04-16 15:23:21+00:00 to 2026-07-15 20:53:21+00:00:
Materialization complete.


## Step 6: Fetch Online Features (Simulate Serving)

Now we pretend to be a live inference request: given a `customer_id`, fetch its current feature values in milliseconds from the online store, exactly as a production API would.

In [48]:
sample_customer_ids = training_df["customer_id"].drop_duplicates().head(5).tolist()

online_features = store.get_online_features(
    features=[
        "customer_txn_stats:txn_amount_sum",
        "customer_txn_stats:txn_count",
        "customer_txn_stats:avg_txn_7d",
    ],
    entity_rows=[{"customer_id": cid} for cid in sample_customer_ids],
).to_df()

online_features

,customer_id,txn_count,avg_txn_7d,txn_amount_sum
0,45,2,56.914520,89.377174
1,190,2,52.591393,60.299488
2,108,2,55.823036,54.524525
3,132,1,48.179783,53.150349
4,46,3,37.635975,43.017948


In [50]:
# Run the model on features pulled straight from the ONLINE store — this is what
# a real-time inference service would do.
X_serve = online_features[feature_cols].fillna(0)
predictions = model.predict(X_serve)

result = online_features[["customer_id"]].copy()
result["predicted_high_spender"] = predictions
result

,customer_id,predicted_high_spender
0,45,1
1,190,1
2,108,1
3,132,0
4,46,0


## Step 7: Consistency Check — Offline vs. Online

This is the core promise of a feature store: the **same customer** should have (approximately) the **same feature values** whether we read them from the offline store or the online store, because both were populated from the one shared feature definition.

Small floating-point rounding differences are expected (float32 storage); large, systematic differences would indicate a skew bug.

In [53]:
# Most recent offline value per sampled customer
latest_offline = (
    customer_features.sort_values("event_timestamp")
    .groupby("customer_id")
    .tail(1)
)
latest_offline = latest_offline[latest_offline["customer_id"].isin(sample_customer_ids)]
latest_offline = latest_offline[["customer_id", "avg_txn_7d"]].rename(
    columns={"avg_txn_7d": "avg_txn_7d_offline"}
)

comparison = online_features[["customer_id", "avg_txn_7d"]].rename(
    columns={"avg_txn_7d": "avg_txn_7d_online"}
)
comparison = comparison.merge(latest_offline, on="customer_id")
comparison["difference"] = (
    comparison["avg_txn_7d_online"] - comparison["avg_txn_7d_offline"]
).abs()

comparison

,customer_id,avg_txn_7d_online,avg_txn_7d_offline,difference
0,45,56.914520,56.914522,1.733808e-06
1,190,52.591393,52.591392,1.000224e-06
2,108,55.823036,55.823037,5.322007e-07
3,132,48.179783,48.179782,1.123775e-06
4,46,37.635975,37.635973,1.860964e-06


If the `difference` column is close to zero for every row, the offline and online layers agree — no training-serving skew. That's the guarantee a feature store is designed to give you.

## Stretch Goal: Introduce a Skew Bug (and Catch It)

Let's simulate the exact bug from the session slides: imagine the **online serving path** was implemented separately (e.g. by another team, in another language) and rounds `avg_txn_7d` differently — say, it truncates to whole rupees instead of keeping decimal precision.

We'll inject that bug manually here, then re-run the same consistency check to show how it surfaces the problem immediately.

In [57]:
# Simulate a buggy "online" computation that truncates to whole numbers
# (a stand-in for: different code path, different rounding logic, etc.)
buggy_online = online_features.copy()
buggy_online["avg_txn_7d"] = buggy_online["avg_txn_7d"].apply(np.floor)

buggy_comparison = buggy_online[["customer_id", "avg_txn_7d"]].rename(
    columns={"avg_txn_7d": "avg_txn_7d_online_buggy"}
)
buggy_comparison = buggy_comparison.merge(latest_offline, on="customer_id")
buggy_comparison["difference"] = (
    buggy_comparison["avg_txn_7d_online_buggy"] - buggy_comparison["avg_txn_7d_offline"]
).abs()

buggy_comparison

,customer_id,avg_txn_7d_online_buggy,avg_txn_7d_offline,difference
0,45,56.0,56.914522,0.914522
1,190,52.0,52.591392,0.591392
2,108,55.0,55.823037,0.823037
3,132,48.0,48.179782,0.179782
4,46,37.0,37.635973,0.635973


Notice the `difference` column is now consistently non-trivial (up to ~1.0) rather than near-zero. In a real system, this kind of check — run continuously as a monitor — is exactly how teams catch training-serving skew before it silently degrades a model in production, which is the failure story we opened the session with.

## Wrap-Up

In this exercise we:

- Defined features once (Entity + FeatureView) instead of duplicating logic across teams
- Retrieved point-in-time correct historical features for training (offline store)
- Materialized features into a low-latency online store
- Retrieved online features to simulate real-time serving
- Verified offline/online consistency, then deliberately broke it to see what skew looks like in practice

This mirrors, at small scale, exactly what Feast, Tecton, and Hopsworks do in production — the core architecture (offline store, online store, single feature definition, materialization job) is the same regardless of which tool or scale you're operating at.